In [1]:
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import cuda
from torch.backends import mps
from torchvision.datasets import ImageFolder
from torchvision.transforms import Lambda
from torchvision.transforms.v2 import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomRotation,
    Resize,
    ToDtype,
    ToImage,
)

from util.image_experiment_runner import run_image_experiment


In [2]:
# Data root for BACH Photos dataset
bach_root = os.path.join("datasets", "ICIAR2018_BACH_Challenge", "Photos")

train_transforms = Compose([
    ToImage(),
    ToDtype(torch.float32, scale=True),
    Resize((224, 224)),
    RandomRotation(10),
    RandomHorizontalFlip(),
    Normalize(mean=[0.5] * 3, std=[0.5] * 3),
])

test_transforms = Compose([
    ToImage(),
    ToDtype(torch.float32, scale=True),
    Resize((224, 224)),
    Normalize(mean=[0.5] * 3, std=[0.5] * 3),
])

full_dataset = ImageFolder(root=bach_root, transform=train_transforms)
full_targets = np.array(full_dataset.targets)

train_idx, test_idx = train_test_split(
    np.arange(len(full_dataset)),
    test_size=0.33,
    stratify=full_targets,
    random_state=42,
    shuffle=True,
)

train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
train_dataset.targets = np.array([full_targets[i] for i in train_idx])

# Use test transforms for evaluation
full_dataset_test = ImageFolder(root=bach_root, transform=test_transforms)
test_dataset = torch.utils.data.Subset(full_dataset_test, test_idx)
test_dataset.targets = np.array([full_targets[i] for i in test_idx])

num_classes = len(full_dataset.classes)
device = torch.device("cuda" if cuda.is_available() else "mps" if mps.is_available() else "cpu")
print(f"Classes: {num_classes}, Device: {device}")
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")


Classes: 4, Device: cpu
Train samples: 268, Test samples: 132


In [ ]:
data_type = "bach"
run_image_experiment(device, num_classes, data_type, train_dataset, test_dataset,
                     optimizer=torch.optim.Adam, lr=0.001, weight_decay=0,
                     batch_size=128, es_patience=40, lr_patience=10, lr_factor=0.5,
                     lr_monitor="train_loss", stratified=False,
                     use_determinism=False, shuffle=False, aps_randomized=True)